In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
import spacy
import re
import contractions
from textblob import TextBlob
from langchain_google_genai import ChatGoogleGenerativeAI


c:\Users\ashmi\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\ashmi\AppData\Local\Temp\ipykernel_848\2073439625.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


### 1,Load the document(.txt)

In [2]:
data=open(r'C:\Users\ashmi\Desktop\Gen AI\RAG\data.txt').read()

### 2,Text Normalization

Converting all characters to lowercase

In [3]:
data=data.lower()

Removing extra space

In [4]:
data=re.sub(r'\s{2,}',' ',data)


Contractions

In [5]:
data=contractions.fix(data)


Removing punctuations and special character

In [6]:
data=re.sub(r'[^0-9a-z\s]','',data)

Correcting the words

In [7]:
# values=TextBlob(data).correct().raw_sentences
# data=" ".join(values)

### 3,Tokenization

*Spacy Lemmantization*


In [8]:
nlp=spacy.load('en_core_web_sm') 
tokens=nlp(data)  
updated_tokens=[token.lemma_ for token in tokens if not token.is_stop]

data=' '.join(updated_tokens).strip()


### 4,Chunking

In [9]:
text_splitters=RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=40
)

chunks=text_splitters.create_documents([data])

*Inserting the informations inside this metadata or dictionary*

In [10]:
chunks[0].metadata={'file_name':'data.txt'}


### 5,Chunk Embeddings

*Chunks to vectors*

In [11]:
embedding_model=HuggingFaceEmbeddings(
     model_name='sentence-transformers/all-miniLM-L6-V2'
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5684.24it/s]


*Vector Database*

In [12]:
vectordb=FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)
vectordb

### 6,Retrival

*Retrival Groundtruth*

In [13]:
rel_chunk={
    'What is Machine Learning?':[
       'machine learning important technology industry \n healthcare finance education transportation retail entertainment cybersecurity \n basic idea machine learning simple',
       'skill contribute successful machine learning project machine learning continue evolve rapidly \n new algorithm architecture introduce regularly \n fundamental principle remain \n good datum essential',
       'machine learning branch artificial intelligence focus enable computer learn pattern datum \n instead explicitly program rule machine learning algorithm learn example use example prediction',
    ]
}

*Retrival*

In [14]:
user_query='What is Machine Learning?'
r_chunks=vectordb.similarity_search(user_query) 

In [15]:
r_chunks=[chunk.page_content for  chunk in r_chunks]
r_chunks

['machine learning important technology industry \n healthcare finance education transportation retail entertainment cybersecurity \n basic idea machine learning simple',
 'skill contribute successful machine learning project machine learning continue evolve rapidly \n new algorithm architecture introduce regularly \n fundamental principle remain \n good datum essential',
 'machine learning branch artificial intelligence focus enable computer learn pattern datum \n instead explicitly program rule machine learning algorithm learn example use example prediction',
 'basic idea machine learning simple \n provide datum algorithm allow algorithm learn pattern use train model prediction new datum \n machine learning divide major category']

### 7,Retrival Evaluation

**Recall@K**

In [16]:
def recall_at_k(retrived,relevant,k:int) -> float:
    retrived_k=set(retrived[:k])
    relevant_set=set(relevant)
    hits=retrived_k & relevant_set
    return len(hits) / len(relevant_set) if relevant_set else 0.0
recall_at_k(list(r_chunks),rel_chunk[user_query],4)


1.0

**Precision@K**

In [17]:
def precision_at_k(retrived,relevant,k:int) -> float:
    retrived_k=set(retrived[:k])
    relevant_set=set(relevant)
    hits=retrived_k & relevant_set
    return len(hits) / k
precision_at_k(list(r_chunks),rel_chunk[user_query],4)


0.75

**Reciprocal Rank**

In [18]:
def reciprocal_rank(retrieved, relevant) -> float:
    relevant_set = set(relevant)
    for rank, doc_id in enumerate(retrieved, start=1):
        if doc_id in relevant_set:
            return 1.0 / rank
    return 0.0  
reciprocal_rank(list(r_chunks),rel_chunk[user_query])

1.0

**Hit Rate**

In [19]:
def hit_rate(retrived,relevant):
    relevant_chunks=set(relevant)
    retrived_chunks=set(retrived)
    hits=relevant_chunks & retrived_chunks
    return 1.0 if len(hits)>=1 else 0.0
hit_rate(list(r_chunks),rel_chunk[user_query])

1.0